# Evaluation, profiling and report artifacts

EN3150 Assignment 03 - Xception. Tasks 4 and 6.

This notebook exposes the evaluation, MAC/parameter profiling, plotting and LaTeX table generators. Its workflow validates recorded artifacts and regenerates tables and figures. It reads the supplied canonical results; fresh rerun_* experiments are kept separate until deliberately selected for a new comparison.

## Running this notebook

Use Python 3.10+ with the project dependencies. Extract the whole ZIP and open this notebook beside raw_code/. On Colab, upload and extract the complete ZIP before running. Set INSTALL_DEPENDENCIES = True in the setup cell for a fresh runtime. Run cells in order with a fresh kernel for each notebook.

The writefile cells below contain the original Python source, including comments. They save that code into the bundled package so relative imports and CLI entry points behave exactly as in the repository. The complete package is already included: the notebooks can be run independently.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Extract the entire ZIP first. This also works from notebooks/submission in Git.
start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT = next((base for parent in candidates
                for base in (parent / "raw_code", parent)
                if (base / "src/pr_edge").is_dir()
                and (base / "configs/base.yaml").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Extract the complete code ZIP, including raw_code, first.")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
sys.path.insert(0, str(PROJECT))

# Enable once in a fresh Python 3.10+ / Colab environment (requires internet).
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

def run_script(script, *args):
    subprocess.run([sys.executable, str(PROJECT / script), *map(str, args)],
                   cwd=PROJECT, check=True)

print("Project:", PROJECT)


## Source: `src/pr_edge/evaluate.py`


In [ ]:
%%writefile src/pr_edge/evaluate.py
"""Test-set evaluation: accuracy, confusion matrix, precision and recall.

Every metric the brief's task 4 names. Both macro-averaged and per-class are
reported: with 10 EuroSAT classes the per-class breakdown is what makes the
confusion matrix discussion substantive rather than decorative.

Scope note: #1 provides a working implementation so ``--smoke`` emits a
schema-conformant ``metrics.json`` in CI. #4 owns the hand-computed
verification tests and any hardening. See the PR discussion on #1.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from .config import Config
from .data import CLASS_NAMES

# Rows are TRUE labels, columns are PREDICTIONS, both ordered by CLASS_NAMES.
# Written into every metrics.json so a figure can never silently transpose it.
CONFUSION_CONVENTION = "rows=true, cols=predicted, order=class_names"


def _confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray, n: int) -> np.ndarray:
    matrix = np.zeros((n, n), dtype=np.int64)
    np.add.at(matrix, (y_true, y_pred), 1)
    return matrix


def _per_class_scores(matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Precision, recall, F1 and support from a confusion matrix.

    A class the model never predicts has an undefined precision (0/0). We
    report 0.0 rather than NaN so the JSON stays numeric and the report tables
    stay readable - but note it in the discussion if it happens, because a
    never-predicted class is a real finding about class imbalance, not a
    rounding detail.
    """
    tp = np.diag(matrix).astype(np.float64)
    predicted = matrix.sum(axis=0).astype(np.float64)  # column sums
    actual = matrix.sum(axis=1).astype(np.float64)     # row sums

    with np.errstate(divide="ignore", invalid="ignore"):
        precision = np.where(predicted > 0, tp / predicted, 0.0)
        recall = np.where(actual > 0, tp / actual, 0.0)
        denom = precision + recall
        f1 = np.where(denom > 0, 2 * precision * recall / denom, 0.0)

    return precision, recall, f1, actual.astype(np.int64)


@torch.no_grad()
def _predict(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple[np.ndarray, np.ndarray]:
    model.eval().to(device)
    trues: list[np.ndarray] = []
    preds: list[np.ndarray] = []

    for images, labels in loader:
        logits = model(images.to(device, non_blocking=True))
        preds.append(logits.argmax(dim=1).cpu().numpy())
        trues.append(labels.numpy())

    return np.concatenate(trues), np.concatenate(preds)


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    cfg: Config,
    device: torch.device,
    out_dir: Path | None = None,
    split: str = "test",
) -> dict:
    """Evaluate a model and optionally write ``metrics.json``.

    The test set is touched exactly once, here, after training is complete.

    Confusion matrix convention: ``confusion_matrix[i][j]`` counts samples
    whose TRUE class is ``class_names[i]`` and whose PREDICTED class is
    ``class_names[j]``. Rows are true labels, columns are predictions - the
    same layout as ``docs/slides/Learning from data.pdf`` p31-p32. So a row
    sum is a class's support (used for recall) and a column sum is how often
    the class was predicted (used for precision).

    Precision, recall and F1 that would be 0/0 (a class never predicted, or
    absent from the labels) are reported as 0.0, never NaN.
    """
    y_true, y_pred = _predict(model, loader, device)

    names = CLASS_NAMES[: cfg.model.num_classes]
    matrix = _confusion_matrix(y_true, y_pred, len(names))
    precision, recall, f1, support = _per_class_scores(matrix)

    metrics = {
        "run_id": cfg.run_id,
        "split": split,
        "num_samples": int(len(y_true)),
        "class_names": names,
        "accuracy": float((y_true == y_pred).mean()),
        # Macro averages weight every class equally, which is the right choice
        # when the report discusses per-class behaviour. EuroSAT is close to
        # balanced, so macro and micro will be similar, but macro is the one
        # that stays honest if a class collapses.
        "macro_precision": float(precision.mean()),
        "macro_recall": float(recall.mean()),
        "macro_f1": float(f1.mean()),
        "per_class": {
            name: {
                "precision": float(precision[i]),
                "recall": float(recall[i]),
                "f1": float(f1[i]),
                "support": int(support[i]),
            }
            for i, name in enumerate(names)
        },
        "confusion_matrix": matrix.tolist(),
        "confusion_matrix_convention": CONFUSION_CONVENTION,
    }

    if out_dir is not None:
        (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    return metrics


## Source: `src/pr_edge/profile.py`


In [ ]:
%%writefile src/pr_edge/profile.py
"""Model resource profiling: parameters, disk size, MACs, epoch timing.

Supplies the resource half of every comparison table in the report. The MAC
count is derived per layer with forward hooks rather than read from a
black-box library, because #11 has to cross-check these numbers against a hand
derivation and an unexplainable number is useless in a report.

Scope note: #1 provides a working implementation so ``--smoke`` emits a
schema-conformant ``profile.json``. #5 owns the CNNs.pdf verification test
(~590k standard vs ~67.8k separable ops) and any refinement. See the PR
discussion on #1.
"""

from __future__ import annotations

import io
import json
from pathlib import Path

import torch
import torch.nn as nn

from .config import Config
from .device import device_name


def count_parameters(model: nn.Module) -> tuple[int, int]:
    """Return ``(total, trainable)`` parameter counts."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def state_dict_bytes(model: nn.Module) -> int:
    """Serialised size of the state dict, measured rather than estimated.

    Saving to an in-memory buffer keeps this side-effect free, and it counts
    the real pickle overhead that an on-disk checkpoint would carry - which is
    what "estimated model size on disk" in the brief actually means.
    """
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes


def count_macs(model: nn.Module, input_shape: tuple[int, ...]) -> tuple[int, dict[str, int]]:
    """Multiply-accumulates for ONE forward pass of ONE image.

    Batch size is forced to 1 so the number is per-image regardless of the
    config's batch size. Bias additions are excluded, matching the convention
    in docs/slides/CNNs.pdf: a k x k convolution producing H x W x C_out from
    C_in channels costs ``k * k * C_in * C_out * H * W``.

    Grouped convolutions - which is what a depthwise convolution is - divide by
    the group count, which is exactly where Model B's saving comes from.
    """
    per_layer: dict[str, int] = {}
    handles = []

    def conv_hook(name: str):
        def hook(module: nn.Conv2d, _inputs, output: torch.Tensor) -> None:
            out_h, out_w = output.shape[-2:]
            kernel = module.kernel_size[0] * module.kernel_size[1]
            in_per_group = module.in_channels // module.groups
            per_layer[name] = kernel * in_per_group * module.out_channels * out_h * out_w
        return hook

    def linear_hook(name: str):
        def hook(module: nn.Linear, _inputs, _output) -> None:
            per_layer[name] = module.in_features * module.out_features
        return hook

    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            handles.append(module.register_forward_hook(conv_hook(name)))
        elif isinstance(module, nn.Linear):
            handles.append(module.register_forward_hook(linear_hook(name)))

    device = next(model.parameters()).device
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            model(torch.zeros(1, *input_shape[1:], device=device))
    finally:
        for handle in handles:
            handle.remove()
        model.train(was_training)

    return sum(per_layer.values()), per_layer


# Parameter-free layers the table leaves out, as Keras's model.summary() folds
# activations into the layer they follow (CNNs.pdf p54 lists no ReLU rows).
_TABLE_SKIP = (nn.ReLU, nn.ReLU6, nn.LeakyReLU, nn.Dropout)


def layer_table(model: nn.Module, input_shape: tuple[int, ...]) -> list[dict]:
    """A ``model.summary()``-style table: one row per layer, in execution order.

    Each row is ``{"name", "type", "output_shape", "params", "kernel_size",
    "groups"}``, where ``output_shape`` excludes the batch dimension, ``params``
    counts only that layer's own weights and biases, and ``kernel_size`` and
    ``groups`` are set for convolutions (``None`` otherwise) so a depthwise
    convolution (groups == in_channels) can be told apart from a standard one. The format mirrors the Keras summary shown
    in CNNs.pdf p54 so the report's architecture tables read like the slides.

    Rows come from a real forward pass of one image, so the shapes are the ones
    the model actually produces rather than ones derived on paper - which is
    what #11's hand derivation is then checked against.
    """
    rows: list[dict] = []
    handles = []

    def hook(name: str):
        def record(module: nn.Module, _inputs, output: torch.Tensor) -> None:
            is_conv = isinstance(module, nn.Conv2d)
            rows.append({
                "name": name,
                "type": type(module).__name__,
                "output_shape": list(output.shape[1:]),
                "params": sum(p.numel() for p in module.parameters(recurse=False)),
                "kernel_size": list(module.kernel_size) if is_conv else None,
                "groups": module.groups if is_conv else None,
            })
        return record

    for name, module in model.named_modules():
        is_leaf = next(module.children(), None) is None
        if name and is_leaf and not isinstance(module, _TABLE_SKIP):
            handles.append(module.register_forward_hook(hook(name)))

    device = next(model.parameters()).device
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            model(torch.zeros(1, *input_shape[1:], device=device))
    finally:
        for handle in handles:
            handle.remove()
        model.train(was_training)

    return rows


def profile_model(
    model: nn.Module,
    cfg: Config,
    mean_epoch_seconds: float,
    device: torch.device,
    out_dir: Path | None = None,
) -> dict:
    """Collect every resource metric and optionally write ``profile.json``."""
    total, trainable = count_parameters(model)
    size_bytes = state_dict_bytes(model)
    input_shape = (1, 3, cfg.data.input_size, cfg.data.input_size)
    macs, per_layer = count_macs(model, input_shape)

    profile = {
        "run_id": cfg.run_id,
        "total_params": total,
        "trainable_params": trainable,
        "disk_size_bytes": size_bytes,
        # Both units are emitted: the brief wants KB for the custom models and
        # MB for the SOTA models, and the magnitude gap between them is itself
        # a point the report makes.
        "disk_size_kb": round(size_bytes / 1024, 2),
        "disk_size_mb": round(size_bytes / 1024**2, 4),
        "macs": macs,
        "macs_per_layer": per_layer,
        "input_shape": list(input_shape),
        "mean_epoch_seconds": round(mean_epoch_seconds, 4),
        "device": device.type,
        # Epoch time is a graded column and is meaningless without the
        # hardware it was measured on.
        "device_name": device_name(device),
    }

    if out_dir is not None:
        (out_dir / "profile.json").write_text(json.dumps(profile, indent=2), encoding="utf-8")

    return profile


## Source: `src/pr_edge/schemas.py`


In [ ]:
%%writefile src/pr_edge/schemas.py
"""Machine-checkable definitions of the three result-file schemas.

docs/interfaces.md section 3 is the prose version; this is the enforceable one.
``tests/test_schemas.py`` validates every committed results file against these,
and CI runs it on every pull request.

Deliberately dependency-free - no jsonschema, no pydantic. The schemas are
simple enough that a 60-line validator is clearer than a library, and one fewer
pinned dependency is one fewer thing to break on a Colab runtime.
"""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

Number = (int, float)

HISTORY_FIELDS: dict[str, tuple] = {
    "epoch": (int,),
    "train_loss": Number,
    "train_acc": Number,
    "val_loss": Number,
    "val_acc": Number,
    "epoch_seconds": Number,
    "lr": Number,
}

METRICS_FIELDS: dict[str, tuple] = {
    "run_id": (str,),
    "split": (str,),
    "num_samples": (int,),
    "class_names": (list,),
    "accuracy": Number,
    "macro_precision": Number,
    "macro_recall": Number,
    "macro_f1": Number,
    "per_class": (dict,),
    "confusion_matrix": (list,),
    "confusion_matrix_convention": (str,),
}

PER_CLASS_FIELDS: dict[str, tuple] = {
    "precision": Number,
    "recall": Number,
    "f1": Number,
    "support": (int,),
}

PROFILE_FIELDS: dict[str, tuple] = {
    "run_id": (str,),
    "total_params": (int,),
    "trainable_params": (int,),
    "disk_size_bytes": (int,),
    "disk_size_kb": Number,
    "disk_size_mb": Number,
    "macs": (int,),
    "macs_per_layer": (dict,),
    "input_shape": (list,),
    "mean_epoch_seconds": Number,
    "device": (str,),
    "device_name": (str,),
}

SELECTION_FIELDS: dict[str, tuple] = {
    "criterion": (str,),
    "best_epoch": (int,),
    "best_val_loss": Number,
    "best_val_acc": Number,
    "epochs_trained": (int,),
}


class SchemaError(ValueError):
    """A result file does not match its frozen schema."""


def _check_fields(obj: Any, spec: dict[str, tuple], where: str) -> list[str]:
    errors: list[str] = []
    if not isinstance(obj, dict):
        return [f"{where}: expected an object, got {type(obj).__name__}"]
    for key, types in spec.items():
        if key not in obj:
            errors.append(f"{where}: missing required key '{key}'")
        elif not isinstance(obj[key], types) or isinstance(obj[key], bool):
            got = type(obj[key]).__name__
            want = "/".join(t.__name__ for t in types)
            errors.append(f"{where}: '{key}' should be {want}, got {got}")
    for key in obj:
        if key not in spec:
            errors.append(f"{where}: unexpected key '{key}'")
    return errors


def validate_history(data: Any) -> list[str]:
    """Return a list of schema violations; empty means valid."""
    if not isinstance(data, list):
        return ["history: expected a JSON array of epoch records"]
    if not data:
        return ["history: must contain at least one epoch"]
    errors: list[str] = []
    for i, rec in enumerate(data):
        errors += _check_fields(rec, HISTORY_FIELDS, f"history[{i}]")
    epochs = [r.get("epoch") for r in data if isinstance(r, dict)]
    if epochs and epochs != list(range(1, len(epochs) + 1)):
        errors.append("history: 'epoch' must be 1-indexed and contiguous")
    return errors


def validate_metrics(data: Any) -> list[str]:
    errors = _check_fields(data, METRICS_FIELDS, "metrics")
    if errors or not isinstance(data, dict):
        return errors

    names = data["class_names"]
    matrix = data["confusion_matrix"]
    n = len(names)

    if len(matrix) != n or any(not isinstance(r, list) or len(r) != n for r in matrix):
        errors.append(f"metrics: confusion_matrix must be {n}x{n} to match class_names")

    missing = set(names) - set(data["per_class"])
    if missing:
        errors.append(f"metrics: per_class missing entries for {sorted(missing)}")
    extra = set(data["per_class"]) - set(names)
    if extra:
        errors.append(f"metrics: per_class has entries not in class_names: {sorted(extra)}")

    for cls, entry in data["per_class"].items():
        errors += _check_fields(entry, PER_CLASS_FIELDS, f"metrics.per_class['{cls}']")

    # per_class is keyed by class NAME, never by index - a figure labelled from
    # integer keys is how a confusion matrix ends up silently transposed.
    if any(str(k).isdigit() for k in data["per_class"]):
        errors.append("metrics: per_class must be keyed by class name, not index")

    return errors


def validate_profile(data: Any) -> list[str]:
    errors = _check_fields(data, PROFILE_FIELDS, "profile")
    if errors or not isinstance(data, dict):
        return errors
    if data["trainable_params"] > data["total_params"]:
        errors.append("profile: trainable_params exceeds total_params")
    if len(data["input_shape"]) != 4:
        errors.append("profile: input_shape must be [N, C, H, W]")
    return errors


def validate_selection(data: Any) -> list[str]:
    errors = _check_fields(data, SELECTION_FIELDS, "selection")
    if errors or not isinstance(data, dict):
        return errors
    if data["criterion"] != "val_loss":
        errors.append("selection: criterion must be 'val_loss'")
    if not 1 <= data["best_epoch"] <= data["epochs_trained"]:
        errors.append("selection: best_epoch must lie within 1..epochs_trained")
    return errors


def check_selection_matches_history(selection: Any, history: Any) -> list[str]:
    """The selected epoch must be the one history.json says it should be.

    Re-derives the choice from the recorded validation losses: the first epoch
    holding the lowest finite ``val_loss``. A disagreement means the evaluated
    checkpoint is not the one the report claims, which no single-file schema
    check can detect.
    """
    if not isinstance(selection, dict) or not isinstance(history, list) or not history:
        return []
    losses = [
        (r["val_loss"], r["epoch"]) for r in history
        if isinstance(r, dict) and isinstance(r.get("val_loss"), Number)
        and math.isfinite(r["val_loss"])
    ]
    errors: list[str] = []
    if selection.get("epochs_trained") != len(history):
        errors.append(f"selection: epochs_trained is {selection.get('epochs_trained')} "
                      f"but history.json has {len(history)} epochs")
    if losses:
        expected = min(losses)[1]  # ties resolve to the earlier epoch
        if selection.get("best_epoch") != expected:
            errors.append(f"selection: best_epoch is {selection.get('best_epoch')} but "
                          f"history.json's lowest val_loss is at epoch {expected}")
    return errors


VALIDATORS = {
    "history.json": validate_history,
    "metrics.json": validate_metrics,
    "profile.json": validate_profile,
    "selection.json": validate_selection,
}


def validate_file(path: Path) -> list[str]:
    """Validate one result file, dispatching on its filename."""
    validator = VALIDATORS.get(path.name)
    if validator is None:
        return [f"{path}: no schema registered for '{path.name}'"]
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        return [f"{path}: invalid JSON - {exc}"]
    return [f"{path.parent.name}/{path.name} {e}" for e in validator(data)]


def validate_run_dir(directory: Path) -> list[str]:
    """Validate a complete run directory, including that nothing is missing."""
    errors: list[str] = []
    for name in VALIDATORS:
        path = directory / name
        if not path.exists():
            errors.append(f"{directory.name}: missing {name}")
        else:
            errors += validate_file(path)
    if not (directory / "config.yaml").exists():
        errors.append(f"{directory.name}: missing config.yaml")

    if not errors:
        history = json.loads((directory / "history.json").read_text(encoding="utf-8"))
        selection = json.loads((directory / "selection.json").read_text(encoding="utf-8"))
        errors += [f"{directory.name}/{e}"
                   for e in check_selection_matches_history(selection, history)]
    return errors


## Source: `src/pr_edge/plots.py`


In [ ]:
%%writefile src/pr_edge/plots.py
"""Report figures: loss curves and confusion matrices (issue #17).

Every figure regenerates from ``results/*/`` with one command. Nothing is
hand-edited, so rerunning an experiment updates the report automatically.
"""

from __future__ import annotations

import json
import math
from pathlib import Path

import matplotlib
import numpy as np

from .paths import RESULTS_DIR
from .schemas import validate_history, validate_metrics

# Figure generation runs in CI and Colab as well as on desktops.  Selecting a
# non-interactive backend before importing pyplot keeps it independent of a
# display server and prevents a report build from opening GUI windows.
matplotlib.use("Agg")
from matplotlib import pyplot as plt  # noqa: E402


def _read_json(path: Path) -> object:
    if not path.exists():
        raise FileNotFoundError(f"missing result artifact: {path}")
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"invalid JSON in {path}: {exc}") from exc


def _run_label(run_id: str) -> str:
    labels = {
        "model_a": "Model A",
        "model_a_100": "Model A (100 epochs)",
        "model_b": "Model B",
        "model_b_100": "Model B (100 epochs)",
        # SOTA runs (#20, #54): name the backbone policy, since the frozen and
        # fine-tuned runs share an architecture and differ only in training.
        "mobilenetv2_64": "MobileNetV2 64x64 (frozen)",
        "mobilenetv2_64_ft": "MobileNetV2 64x64 (fine-tuned)",
        "mobilenetv2_224": "MobileNetV2 224x224 (frozen)",
        "mobilenetv2_224_ft": "MobileNetV2 224x224 (fine-tuned, batch 32)",
        "mobilenetv2_224_ft_b128": "MobileNetV2 224x224 (fine-tuned)",
        # EfficientNet-B0 (#21, #64): the same matrix as MobileNetV2.
        "efficientnetb0_64": "EfficientNet-B0 64x64 (frozen)",
        "efficientnetb0_64_ft": "EfficientNet-B0 64x64 (fine-tuned)",
        "efficientnetb0_224": "EfficientNet-B0 224x224 (frozen)",
        "efficientnetb0_224_ft": "EfficientNet-B0 224x224 (fine-tuned, batch 32)",
        "efficientnetb0_224_ft_b128": "EfficientNet-B0 224x224 (fine-tuned)",
    }
    return labels.get(run_id, run_id.replace("_", " ").title())


def _save(fig, out: Path) -> Path:
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return out


def plot_loss_curves(
    run_ids: list[str],
    out: Path,
    title: str | None = None,
    reference_epoch: int | None = None,
) -> Path:
    """Training and validation loss against epoch, one series pair per run.

    Each run gets its own axis so its train/validation pair stays readable.
    The axes share one y-scale, preventing independent autoscaling from making
    a small loss difference look as large as a large one.
    """
    if not run_ids:
        raise ValueError("plot_loss_curves requires at least one run_id")

    histories: list[tuple[str, list[dict]]] = []
    for run_id in run_ids:
        path = RESULTS_DIR / run_id / "history.json"
        data = _read_json(path)
        errors = validate_history(data)
        if errors:
            raise ValueError(f"invalid {run_id}/history.json: {'; '.join(errors)}")
        histories.append((run_id, data))

    fig, axes = plt.subplots(
        1,
        len(histories),
        figsize=(6.2 * len(histories), 6.2),
        sharey=True,
        squeeze=False,
    )
    axes = axes[0]

    all_losses = [
        float(record[field])
        for _, history in histories
        for record in history
        for field in ("train_loss", "val_loss")
    ]
    finite_losses = [value for value in all_losses if math.isfinite(value)]
    if not finite_losses:
        plt.close(fig)
        raise ValueError("loss histories contain no finite train or validation loss")
    upper = max(finite_losses)
    lower = min(0.0, min(finite_losses))
    padding = max((upper - lower) * 0.05, 0.05)

    for axis, (run_id, history) in zip(axes, histories, strict=True):
        epochs = [record["epoch"] for record in history]
        axis.plot(
            epochs,
            [record["train_loss"] for record in history],
            color="#2563EB",
            linewidth=2.1,
            label="Training",
        )
        axis.plot(
            epochs,
            [record["val_loss"] for record in history],
            color="#DC2626",
            linewidth=2.1,
            label="Validation",
        )
        if reference_epoch is not None:
            axis.axvline(
                reference_epoch,
                color="#111827",
                linestyle="--",
                linewidth=1.5,
                label=f"Original stop ({reference_epoch})",
            )
        axis.set_title(_run_label(run_id), fontweight="bold")
        axis.set_xlabel("Epoch")
        axis.set_ylim(lower, upper + padding)
        axis.grid(axis="y", color="#D1D5DB", linewidth=0.7, alpha=0.8)
        axis.legend(frameon=False)

    axes[0].set_ylabel("Cross-entropy loss")
    if title:
        fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout()
    return _save(fig, out)


def plot_confusion_matrix(run_id: str, out: Path, normalize: bool = True) -> Path:
    """Labelled confusion matrix heatmap for one run.

    Colour is normalised by true-class row, making each diagonal cell that
    class's recall.  Every cell is annotated with its raw count so the figure
    preserves the sample support and exact mistakes behind the proportions.
    """
    path = RESULTS_DIR / run_id / "metrics.json"
    data = _read_json(path)
    errors = validate_metrics(data)
    if errors:
        raise ValueError(f"invalid {run_id}/metrics.json: {'; '.join(errors)}")

    names = data["class_names"]
    counts = np.asarray(data["confusion_matrix"], dtype=np.int64)
    row_totals = counts.sum(axis=1, keepdims=True)
    if normalize:
        values = np.divide(
            counts,
            row_totals,
            out=np.zeros_like(counts, dtype=np.float64),
            where=row_totals != 0,
        )
        colorbar_label = "Proportion of true class"
        vmax = 1.0
    else:
        values = counts
        colorbar_label = "Samples"
        vmax = None

    fig, axis = plt.subplots(figsize=(10.5, 8.4))
    image = axis.imshow(values, cmap="Blues", vmin=0, vmax=vmax, aspect="equal")
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    colorbar.set_label(colorbar_label)

    positions = np.arange(len(names))
    axis.set_xticks(positions, labels=names, rotation=45, ha="right", rotation_mode="anchor")
    axis.set_yticks(positions, labels=names)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
    axis.set_title(f"{_run_label(run_id)} test confusion matrix", fontweight="bold")

    threshold = (float(values.max()) / 2.0) if values.size else 0.0
    for row in range(len(names)):
        for column in range(len(names)):
            axis.text(
                column,
                row,
                f"{counts[row, column]:d}",
                ha="center",
                va="center",
                fontsize=7.5,
                color="white" if values[row, column] > threshold else "#111827",
            )

    # Keep the first row at the top after explicitly setting the image bounds.
    axis.set_xlim(-0.5, len(names) - 0.5)
    axis.set_ylim(len(names) - 0.5, -0.5)
    fig.tight_layout()
    return _save(fig, out)


## Source: `src/pr_edge/tables.py`


In [ ]:
%%writefile src/pr_edge/tables.py
"""LaTeX table generation from result artifacts.

This module is the only path by which a number reaches the report:

    results/*/[metrics|profile].json -> build_tables.py -> report/tables/*.tex -> \\input{}

Never hand-type a result into LaTeX. If a run is repeated, the report updates
itself; if a number is typed, it silently goes stale and nobody notices until
the viva.

Each function emits a bare ``tabular`` fragment - no ``\\begin{table}``
wrapper. Caption, label and placement belong to the report, not the generator.
"""

from __future__ import annotations

import json
from pathlib import Path

import yaml

from .paths import RESULTS_DIR
from .schemas import validate_metrics, validate_profile, validate_run_dir


# Keras-style layer names, as in the model.summary() table on CNNs.pdf p54, so
# the report's architecture tables use the vocabulary the slides use.
_LAYER_NAMES = {
    "Conv2d": "Conv2D",
    "MaxPool2d": "MaxPooling2D",
    "AdaptiveAvgPool2d": "GlobalAveragePooling2D",
    "Flatten": "Flatten",
    "Linear": "Dense",
}


def _layer_kind(row: dict) -> str:
    """Report name for one layer_table row.

    The two stages of a depthwise separable block are both nn.Conv2d, so they
    are told apart by shape: a grouped convolution is the depthwise step and a
    1x1 convolution is the pointwise step, the two names CNNs.pdf p44 uses.
    """
    if row["type"] == "Conv2d":
        if row["groups"] and row["groups"] > 1:
            return "DepthwiseConv2D"
        if row["kernel_size"] == [1, 1]:
            return "PointwiseConv2D"
    return _LAYER_NAMES.get(row["type"], row["type"])


def _thousands(n: int) -> str:
    """18496 -> 18{,}496 - a LaTeX thousands separator with correct spacing."""
    return f"{n:,}".replace(",", "{,}")


def build_layer_table(config: Path, out: Path) -> Path:
    """Per-layer output shape and parameter count for a custom model.

    Built from the model itself (constructed from ``config``), never from
    hand-typed numbers, so a change to the channel widths updates the report.
    Emits a bare ``tabular`` like every other generator here: layer, output
    shape (C x H x W), parameters, and a total row. Implemented by #9; #10
    reuses it for Model B.
    """
    # Imported here so building an unrelated table never needs torch.
    from .config import load_config
    from .models import build_model
    from .profile import count_parameters, layer_table

    cfg = load_config(config)
    model = build_model(cfg)
    input_shape = (1, 3, cfg.data.input_size, cfg.data.input_size)
    rows = layer_table(model, input_shape)
    total, _ = count_parameters(model)

    kinds = [_layer_kind(row) for row in rows]
    counters: dict[str, int] = {}
    lines = [
        r"\begin{tabular}{llr}",
        r"\hline",
        r"Layer & Output shape ($C \times H \times W$) & Parameters \\",
        r"\hline",
    ]
    for row, kind in zip(rows, kinds):
        # Number only the layer kinds that repeat: "Conv2D 2", but plain "Flatten".
        counters[kind] = counters.get(kind, 0) + 1
        label = f"{kind} {counters[kind]}" if kinds.count(kind) > 1 else kind
        shape = " \\times ".join(str(d) for d in row["output_shape"])
        lines.append(f"{label} & ${shape}$ & {_thousands(row['params'])} \\\\")
    lines += [
        r"\hline",
        f"Total & & {_thousands(total)} \\\\",
        r"\hline",
        r"\end{tabular}",
    ]

    out.parent.mkdir(parents=True, exist_ok=True)
    # Explicit "\n" so the generated file is byte-identical on every platform.
    out.write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
    return out


def build_dataset_table(manifest: Path, out: Path) -> Path:
    """Per-class image counts in each split, read from the committed manifest.

    Rows follow CLASS_NAMES order, then a total row and a share row that shows
    the realised 70/15/15 proportions. Built from data/splits.csv itself - the
    file every experiment reads - so the table can only describe the partition
    that was actually used. Implemented by #24.
    """
    from .data import CLASS_NAMES, SPLIT_FRACTIONS, read_manifest, validate_manifest

    rows = read_manifest(manifest)
    # Refuse to tabulate a manifest the experiments themselves would reject.
    validate_manifest(rows)

    splits = list(SPLIT_FRACTIONS)
    counts = {name: dict.fromkeys(splits, 0) for name in CLASS_NAMES}
    for row in rows:
        counts[row["label"]][row["split"]] += 1
    totals = {split: sum(counts[name][split] for name in CLASS_NAMES) for split in splits}
    grand = sum(totals.values())

    lines = [
        r"\begin{tabular}{lrrrr}",
        r"\toprule",
        r"Class & Train & Validation & Test & Total \\",
        r"\midrule",
    ]
    for name in CLASS_NAMES:
        cells = [_thousands(counts[name][split]) for split in splits]
        lines.append(f"{name} & {' & '.join(cells)} & {_thousands(sum(counts[name].values()))} \\\\")
    lines.append(r"\midrule")
    cells = [_thousands(totals[split]) for split in splits]
    lines.append(f"Total & {' & '.join(cells)} & {_thousands(grand)} \\\\")
    shares = [f"{100 * totals[split] / grand:.1f}\\%" for split in splits]
    lines.append(f"Share & {' & '.join(shares)} & 100.0\\% \\\\")
    lines += [r"\bottomrule", r"\end{tabular}"]

    out.parent.mkdir(parents=True, exist_ok=True)
    # Explicit "\n" so the generated file is byte-identical on every platform.
    out.write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
    return out


def build_custom_comparison(run_ids: list[str], out: Path) -> Path:
    """Model A vs Model B.

    The brief names these columns: total parameter count, estimated model size
    on disk (KB), training time per epoch, and test accuracy. Macro precision
    and recall are added because task 4 requires them too. Implemented by #18.
    """
    if not run_ids:
        raise ValueError("build_custom_comparison requires at least one run_id")

    lines = [
        r"\begin{tabular}{lrrrrrr}",
        r"\hline",
        r"Model & Parameters & Size (KB) & Epoch (s) & Accuracy (\%) & Precision (\%) & Recall (\%) \\",
        r"\hline",
    ]
    for run_id in run_ids:
        directory = RESULTS_DIR / run_id
        paths = {
            "metrics": directory / "metrics.json",
            "profile": directory / "profile.json",
        }
        missing = [path.name for path in paths.values() if not path.is_file()]
        if missing:
            raise FileNotFoundError(f"run {run_id!r} is missing custom table artifact(s): {missing}")

        metrics = json.loads(paths["metrics"].read_text(encoding="utf-8"))
        profile = json.loads(paths["profile"].read_text(encoding="utf-8"))
        errors = validate_metrics(metrics) + validate_profile(profile)
        if errors:
            raise ValueError(f"run {run_id!r} has invalid result artifacts: {'; '.join(errors)}")
        if metrics["run_id"] != run_id or profile["run_id"] != run_id:
            raise ValueError(f"run {run_id!r} contains an artifact with a different run_id")

        label = run_id.replace("_", " ").title()
        size_kb = f"{profile['disk_size_kb']:,.2f}".replace(",", "{,}")
        lines.append(
            f"{label} & {_thousands(profile['total_params'])} & {size_kb} & "
            f"{profile['mean_epoch_seconds']:.2f} & {100 * metrics['accuracy']:.2f} & "
            f"{100 * metrics['macro_precision']:.2f} & {100 * metrics['macro_recall']:.2f} \\\\"
        )
    lines += [r"\hline", r"\end{tabular}"]

    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
    return out


def build_custom_convergence(run_ids: list[str], out: Path) -> Path:
    """Controlled 30-vs-100 epoch comparison for Models A and B.

    Selection fields come from the lowest-validation-loss checkpoint, while
    test metrics and resource measurements come from the matching run
    artifacts.  All rows must share a device because epoch time is otherwise
    not comparable.
    """
    if not run_ids:
        raise ValueError("build_custom_convergence requires at least one run_id")

    rows: list[tuple[str, dict, dict, dict, dict]] = []
    devices: set[str] = set()
    for run_id in run_ids:
        directory = RESULTS_DIR / run_id
        errors = validate_run_dir(directory)
        if errors:
            raise ValueError(f"run {run_id!r} has invalid result artifacts: {'; '.join(errors)}")

        config = yaml.safe_load((directory / "config.yaml").read_text(encoding="utf-8"))
        metrics = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))
        profile = json.loads((directory / "profile.json").read_text(encoding="utf-8"))
        selection = json.loads((directory / "selection.json").read_text(encoding="utf-8"))
        if any(artifact["run_id"] != run_id for artifact in (config, metrics, profile)):
            raise ValueError(f"run {run_id!r} contains an artifact with a different run_id")
        if config["training"]["epochs"] != selection["epochs_trained"]:
            raise ValueError(f"run {run_id!r} config and selection disagree on the epoch budget")
        devices.add(profile["device_name"])
        rows.append((run_id, config, metrics, profile, selection))

    if len(devices) != 1:
        raise ValueError(f"custom convergence rows must share one device, found: {sorted(devices)}")

    lines = [
        r"\begin{tabular}{lrrrrrrrrr}",
        r"\hline",
        (
            r"Model & Epochs & Selected & Best val. loss & Best val. acc. (\%) & "
            r"Test acc. (\%) & Parameters & Size (KB) & MACs & Epoch (s) \\"
        ),
        r"\hline",
    ]
    for _, config, metrics, profile, selection in rows:
        model = {"model_a": "Model A", "model_b": "Model B"}.get(
            config["model"]["name"], config["model"]["name"].replace("_", " ").title()
        )
        size_kb = f"{profile['disk_size_kb']:,.2f}".replace(",", "{,}")
        lines.append(
            f"{model} & {selection['epochs_trained']} & {selection['best_epoch']} & "
            f"{selection['best_val_loss']:.4f} & {100 * selection['best_val_acc']:.2f} & "
            f"{100 * metrics['accuracy']:.2f} & {_thousands(profile['total_params'])} & "
            f"{size_kb} & {_thousands(profile['macs'])} & {profile['mean_epoch_seconds']:.2f} \\\\"
        )
    lines += [r"\hline", r"\end{tabular}"]

    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
    return out


def build_optimizer_comparison(run_ids: list[str], out: Path) -> Path:
    """Adam vs SGD vs SGD with momentum for Task 3.

    Reads the resolved optimizer configuration and validation/test artifacts
    from each run.  Keeping this generated prevents a repeated experiment from
    leaving stale, hand-typed values in the report.
    """
    lines = [
        r"\begin{tabular}{lrrrrrr}",
        r"\hline",
        r"Optimizer & LR & SGD $m$ & Best val. loss & Val. acc. (\%) & Test acc. (\%) & Epoch (s) \\",
        r"\hline",
    ]
    for run_id in run_ids:
        directory = RESULTS_DIR / run_id
        required = ["config.yaml", "metrics.json", "profile.json", "selection.json"]
        missing = [name for name in required if not (directory / name).is_file()]
        if missing:
            raise FileNotFoundError(f"run {run_id!r} is missing optimizer table artifact(s): {missing}")

        config = yaml.safe_load((directory / "config.yaml").read_text(encoding="utf-8"))
        metrics = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))
        profile = json.loads((directory / "profile.json").read_text(encoding="utf-8"))
        selection = json.loads((directory / "selection.json").read_text(encoding="utf-8"))
        optimizer = config["optimizer"]
        name = "Adam" if optimizer["name"] == "adam" else "SGD"
        if optimizer["name"] == "sgd" and optimizer["momentum"] > 0:
            name = "SGD + momentum"
        # This config field is SGD-only. Adam's moment coefficients are not zero.
        momentum = f"{optimizer['momentum']:.1f}" if optimizer["name"] == "sgd" else "--"
        lines.append(
            f"{name} & {optimizer['lr']:.3g} & {momentum} & "
            f"{selection['best_val_loss']:.3f} & {100 * selection['best_val_acc']:.2f} & "
            f"{100 * metrics['accuracy']:.2f} & {profile['mean_epoch_seconds']:.2f} \\\\"
        )
    lines += [r"\hline", r"\end{tabular}"]

    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
    return out


# Report names for the torchvision backbones, keyed by config model.name.
_SOTA_NAMES = {
    "mobilenet_v2": "MobileNetV2",
    "efficientnet_b0": "EfficientNet-B0",
}

# The resolution every primary result uses. Any other input size is the
# supplementary resolution ablation and is grouped below the primary rows.
_PRIMARY_INPUT_SIZE = 64


def build_sota_comparison(run_ids: list[str], out: Path) -> Path:
    """Fine-tuned lightweight SOTA models.

    Sizes in MB here, not KB - the brief asks for MB for these, and the
    magnitude gap against the custom models is itself the point. Implemented
    by #22.

    Columns: model, backbone policy (frozen or fine-tuned, CNNs.pdf p58),
    input resolution, test accuracy, macro precision, macro recall, total and
    trained parameter counts, and size in MB. Rows at the primary 64x64 input
    come first in ``run_ids`` order. Rows at any other input size follow under
    a labelled "Supplementary" rule, so the 224x224 ablation cannot be
    mistaken for a primary result.

    Model name, backbone policy and input size are read from each run's
    resolved ``config.yaml`` - what the run actually did - never parsed out
    of the run ID.
    """
    if not run_ids:
        raise ValueError("build_sota_comparison requires at least one run_id")

    primary, supplementary = [], []
    for run_id in run_ids:
        directory = RESULTS_DIR / run_id
        required = ["config.yaml", "metrics.json", "profile.json"]
        missing = [name for name in required if not (directory / name).is_file()]
        if missing:
            raise FileNotFoundError(f"run {run_id!r} is missing SOTA table artifact(s): {missing}")

        config = yaml.safe_load((directory / "config.yaml").read_text(encoding="utf-8"))
        metrics = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))
        profile = json.loads((directory / "profile.json").read_text(encoding="utf-8"))
        errors = validate_metrics(metrics) + validate_profile(profile)
        if errors:
            raise ValueError(f"run {run_id!r} has invalid result artifacts: {'; '.join(errors)}")
        if config["run_id"] != run_id or metrics["run_id"] != run_id or profile["run_id"] != run_id:
            raise ValueError(f"run {run_id!r} contains an artifact with a different run_id")
        # Every row must describe the held-out test split, never validation.
        if metrics["split"] != "test":
            raise ValueError(f"run {run_id!r} metrics are for split {metrics['split']!r}, not 'test'")

        model_name = config["model"]["name"]
        if model_name not in _SOTA_NAMES:
            raise ValueError(f"run {run_id!r} is not a SOTA model: {model_name!r}")
        backbone = "Frozen" if config["model"]["freeze_backbone"] else "Fine-tuned"
        size = config["data"]["input_size"]

        row = (
            f"{_SOTA_NAMES[model_name]} & {backbone} & ${size}\\times{size}$ & "
            f"{100 * metrics['accuracy']:.2f} & {100 * metrics['macro_precision']:.2f} & "
            f"{100 * metrics['macro_recall']:.2f} & {_thousands(profile['total_params'])} & "
            f"{_thousands(profile['trainable_params'])} & {profile['disk_size_mb']:.2f} \\\\"
        )
        (primary if size == _PRIMARY_INPUT_SIZE else supplementary).append(row)

    # Nine columns do not fit the text width on one header line, so the
    # numeric headers stack name over unit. \shortstack needs no package, and
    # [r] matches the right-aligned numbers beneath it.
    lines = [
        r"\begin{tabular}{lllrrrrrr}",
        r"\toprule",
        r"Model & Backbone & Input & \shortstack[r]{Accuracy\\(\%)} & "
        r"\shortstack[r]{Precision\\(\%)} & \shortstack[r]{Recall\\(\%)} & "
        r"Parameters & Trained & \shortstack[r]{Size\\(MB)} \\",
        r"\midrule",
        *primary,
    ]
    if supplementary:
        lines += [
            r"\midrule",
            r"\multicolumn{9}{l}{\textit{Supplementary: input resolution ablation}} \\",
            *supplementary,
        ]
    lines += [r"\bottomrule", r"\end{tabular}"]

    out.parent.mkdir(parents=True, exist_ok=True)
    # Explicit "\n" so the generated file is byte-identical on every platform.
    out.write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
    return out


def build_final_comparison(run_ids: list[str], out: Path) -> Path:
    """Three-way table for task 6: Model B against both SOTA models.

    Accuracy, parameters, size, MACs and epoch time. Epoch budgets are shown
    because the convergence-controlled Model B run trains longer than SOTA.
    Global TeX macros keep the accompanying discussion's numbers tied to the
    same artifacts, even when the fragment is input inside a table group.
    """
    if len(run_ids) != 3 or len(set(run_ids)) != 3:
        raise ValueError("final comparison requires exactly three distinct run_ids")

    rows = {}
    comparison_keys = []
    for run_id in run_ids:
        directory = RESULTS_DIR / run_id
        errors = validate_run_dir(directory)
        if errors:
            raise ValueError(f"run {run_id!r} has invalid result artifacts: {'; '.join(errors)}")
        config = yaml.safe_load((directory / "config.yaml").read_text(encoding="utf-8"))
        metrics = json.loads((directory / "metrics.json").read_text(encoding="utf-8"))
        profile = json.loads((directory / "profile.json").read_text(encoding="utf-8"))
        selection = json.loads((directory / "selection.json").read_text(encoding="utf-8"))
        if any(artifact["run_id"] != run_id for artifact in (config, metrics, profile)):
            raise ValueError(f"run {run_id!r} contains an artifact with a different run_id")
        if metrics["split"] != "test":
            raise ValueError(f"run {run_id!r} must contain test metrics")
        if config["training"]["smoke"] or config["training"]["epochs"] != selection["epochs_trained"]:
            raise ValueError(f"run {run_id!r} has an invalid epoch budget or is a smoke run")
        if config["data"]["input_size"] != 64 or profile["input_shape"] != [1, 3, 64, 64]:
            raise ValueError(f"run {run_id!r} must use native 64x64 input")

        model = config["model"]["name"]
        if model not in {"model_b", *_SOTA_NAMES} or model in rows:
            raise ValueError("final comparison requires Model B, MobileNetV2 and EfficientNet-B0 once each")
        if config["model"]["freeze_backbone"] or profile["trainable_params"] != profile["total_params"]:
            raise ValueError(f"run {run_id!r} must train all parameters")
        if config["model"]["pretrained"] != (model != "model_b"):
            raise ValueError(f"run {run_id!r} must use scratch Model B or pretrained SOTA")
        if model == "model_b" and profile["total_params"] > 100_000:
            raise ValueError("Model B exceeds the 100k parameter budget")
        if any(profile[key] <= 0 for key in ("disk_size_bytes", "macs", "mean_epoch_seconds")):
            raise ValueError(f"run {run_id!r} must have positive storage, MACs and epoch time")

        # Learning rates and epoch budgets intentionally differ; the data and
        # hardware must agree before training time can be compared.
        comparison_keys.append((
            config["seed"], config["data"]["root"],
            config["data"]["manifest"], config["data"]["norm_stats"],
            config["data"]["batch_size"], config["data"]["augment"],
            config["data"]["num_workers"], profile["device_name"],
            tuple(metrics["class_names"]), metrics["num_samples"],
        ))
        rows[model] = (config, metrics, profile, selection)
    if len(set(comparison_keys)) != 1:
        raise ValueError("final comparison rows must share data, seed, batch settings and device")

    # Use measured bytes for every conversion and ratio: the stored KB/MB
    # values are rounded, and KB elsewhere in this report means 1024 bytes.
    b_config, b_metrics, b_profile, b_selection = rows["model_b"]
    macros = {
        "FinalBAccuracy": f"{100 * b_metrics['accuracy']:.2f}",
        "FinalBSize": f"{b_profile['disk_size_bytes'] / 1024:.2f}",
        "FinalBParams": _thousands(b_profile["total_params"]),
        "FinalBEpochs": str(b_config["training"]["epochs"]),
        "FinalBSelectedEpoch": str(b_selection["best_epoch"]),
    }
    for model, prefix in (("mobilenet_v2", "FinalMobile"), ("efficientnet_b0", "FinalEfficient")):
        config, metrics, profile, _ = rows[model]
        macros.update({
            prefix + "Accuracy": f"{100 * metrics['accuracy']:.2f}",
            prefix + "Gap": f"{100 * (metrics['accuracy'] - b_metrics['accuracy']):.2f}",
            prefix + "StorageRatio": f"{profile['disk_size_bytes'] / b_profile['disk_size_bytes']:.1f}",
            prefix + "MACRatio": f"{profile['macs'] / b_profile['macs']:.2f}",
            # The recorded epoch times only the training pass, excluding
            # validation, final testing, setup and ImageNet pretraining.
            prefix + "TrainingRatio": (
                f"{b_profile['mean_epoch_seconds'] * b_config['training']['epochs'] / (profile['mean_epoch_seconds'] * config['training']['epochs']):.2f}"
            ),
        })
    mobile = rows["mobilenet_v2"]
    efficient = rows["efficientnet_b0"]
    macros["FinalSOTAGap"] = f"{100 * (efficient[1]['accuracy'] - mobile[1]['accuracy']):.2f}"
    macros["FinalSOTAExtraCorrect"] = str(round(
        (efficient[1]["accuracy"] - mobile[1]["accuracy"]) * mobile[1]["num_samples"]
    ))

    lines = [f"\\gdef\\{name}{{{value}}}%" for name, value in macros.items()]
    lines += [
        r"\begin{tabular}{lrrrrrr}",
        r"\toprule",
        r"Model & Epochs & \shortstack[r]{Accuracy\\(\%)} & Parameters & "
        r"\shortstack[r]{Size\\(KiB)} & \shortstack[r]{MACs\\(M/image)} & "
        r"\shortstack[r]{Epoch\\(s)} \\",
        r"\midrule",
    ]
    for model in ("model_b", "mobilenet_v2", "efficientnet_b0"):
        config, metrics, profile, selection = rows[model]
        label = "Model B" if model == "model_b" else _SOTA_NAMES[model]
        size_kib = f"{profile['disk_size_bytes'] / 1024:,.2f}".replace(",", "{,}")
        lines.append(
            f"{label} & {selection['epochs_trained']} & {100 * metrics['accuracy']:.2f} & "
            f"{_thousands(profile['total_params'])} & {size_kib} & "
            f"{profile['macs'] / 1_000_000:.3f} & {profile['mean_epoch_seconds']:.2f} \\\\"
        )
    lines += [r"\bottomrule", r"\end{tabular}"]
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
    return out


## Source: `scripts/build_tables.py`


In [ ]:
%%writefile scripts/build_tables.py
#!/usr/bin/env python
"""Generate the report's LaTeX tables from committed result artifacts.

    python scripts/build_tables.py --all
    python scripts/build_tables.py --table custom

This is the ONLY path by which a number reaches the report. Individual table
builders are implemented by #18, #22, #24 and #25.
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from pr_edge.paths import CONFIGS_DIR, DATA_DIR, REPORT_DIR  # noqa: E402
from pr_edge.tables import (  # noqa: E402
    build_custom_comparison,
    build_custom_convergence,
    build_dataset_table,
    build_final_comparison,
    build_layer_table,
    build_optimizer_comparison,
    build_sota_comparison,
)

TABLES_DIR = REPORT_DIR / "tables"

# Which runs feed which table. Fixed here so no issue has to guess, and so a
# missing run fails loudly rather than producing a table with a gap in it.
BUILDERS = {
    # Architecture tables are built from the configs, not from runs: they need
    # no training, only the model the config describes.
    "model_a_layers": lambda: build_layer_table(CONFIGS_DIR / "model_a.yaml", TABLES_DIR / "model_a_layers.tex"),
    "model_b_layers": lambda: build_layer_table(CONFIGS_DIR / "model_b.yaml", TABLES_DIR / "model_b_layers.tex"),
    "dataset": lambda: build_dataset_table(DATA_DIR / "splits.csv", TABLES_DIR / "dataset.tex"),
    "custom": lambda: build_custom_comparison(["model_a", "model_b"], TABLES_DIR / "custom_comparison.tex"),
    "custom_convergence": lambda: build_custom_convergence(
        ["model_a", "model_a_100", "model_b", "model_b_100"],
        TABLES_DIR / "custom_convergence.tex",
    ),
    "optimizer": lambda: build_optimizer_comparison(
        ["model_b", "model_b_sgd", "model_b_sgdm"], TABLES_DIR / "optimizer_comparison.tex"
    ),
    # Each network as a frozen / fine-tuned pair (CNNs.pdf p58); the fine-tuned
    # run is the primary SOTA result (#22, #54). Each 224x224 pair differs from
    # its 64x64 one only in input size and renders as a supplementary group;
    # the batch-32 records *_224_ft are superseded by the batch-128 *_b128 runs.
    "sota": lambda: build_sota_comparison(
        [
            "mobilenetv2_64",
            "mobilenetv2_64_ft",
            "efficientnetb0_64",
            "efficientnetb0_64_ft",
            "mobilenetv2_224",
            "mobilenetv2_224_ft_b128",
            "efficientnetb0_224",
            "efficientnetb0_224_ft_b128",
        ],
        TABLES_DIR / "sota_comparison.tex",
    ),
    "final": lambda: build_final_comparison(
        ["model_b_100", "mobilenetv2_64_ft", "efficientnetb0_64_ft"], TABLES_DIR / "final_comparison.tex"
    ),
}


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument("--all", action="store_true", help="build every table")
    group.add_argument("--table", choices=sorted(BUILDERS), help="build one table")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    TABLES_DIR.mkdir(parents=True, exist_ok=True)

    names = sorted(BUILDERS) if args.all else [args.table]
    for name in names:
        path = BUILDERS[name]()
        print(f"[tables] {name} -> {path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Source: `scripts/build_figures.py`


In [ ]:
%%writefile scripts/build_figures.py
#!/usr/bin/env python
"""Generate task 4 and task 5 report figures from committed result artifacts.

    python scripts/build_figures.py --all
    python scripts/build_figures.py --figure loss
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from pr_edge.paths import REPORT_DIR  # noqa: E402
from pr_edge.plots import plot_confusion_matrix, plot_loss_curves  # noqa: E402

FIGURES_DIR = REPORT_DIR / "figures"

BUILDERS = {
    "loss": lambda: plot_loss_curves(
        ["model_a", "model_b"],
        FIGURES_DIR / "custom_loss_curves.png",
        title="Custom CNN training and validation loss",
    ),
    "convergence": lambda: plot_loss_curves(
        ["model_a_100", "model_b_100"],
        FIGURES_DIR / "custom_convergence.png",
        title="Custom CNN 100-epoch convergence control",
        reference_epoch=30,
    ),
    "model_a_confusion": lambda: plot_confusion_matrix(
        "model_a", FIGURES_DIR / "model_a_confusion_matrix.png"
    ),
    "model_b_confusion": lambda: plot_confusion_matrix(
        "model_b", FIGURES_DIR / "model_b_confusion_matrix.png"
    ),
    # Task 5 (#20): frozen vs fine-tuned, one figure per input resolution so
    # each pair shares a y-scale and stays readable at report width. All four
    # runs use batch 128; the batch-32 record mobilenetv2_224_ft is not plotted.
    "sota_loss": lambda: plot_loss_curves(
        ["mobilenetv2_64", "mobilenetv2_64_ft"],
        FIGURES_DIR / "sota_loss_curves.png",
        title="MobileNetV2 at 64x64: training and validation loss",
    ),
    "sota_224_loss": lambda: plot_loss_curves(
        ["mobilenetv2_224", "mobilenetv2_224_ft_b128"],
        FIGURES_DIR / "sota_224_loss_curves.png",
        title="MobileNetV2 at 224x224: training and validation loss",
    ),
    # The primary SOTA run: fine-tuned, at the custom models' 64x64 input.
    "mobilenetv2_64_ft_confusion": lambda: plot_confusion_matrix(
        "mobilenetv2_64_ft", FIGURES_DIR / "mobilenetv2_64_ft_confusion_matrix.png"
    ),
    # EfficientNet-B0 (#64): the same frozen-vs-fine-tuned pairs as MobileNetV2,
    # again at batch 128 only (efficientnetb0_224_ft, the batch-32 record, is
    # not plotted), plus a confusion matrix for each fine-tuned resolution.
    "efficientnet_loss": lambda: plot_loss_curves(
        ["efficientnetb0_64", "efficientnetb0_64_ft"],
        FIGURES_DIR / "efficientnet_loss_curves.png",
        title="EfficientNet-B0 at 64x64: training and validation loss",
    ),
    "efficientnet_224_loss": lambda: plot_loss_curves(
        ["efficientnetb0_224", "efficientnetb0_224_ft_b128"],
        FIGURES_DIR / "efficientnet_224_loss_curves.png",
        title="EfficientNet-B0 at 224x224: training and validation loss",
    ),
    "efficientnetb0_64_ft_confusion": lambda: plot_confusion_matrix(
        "efficientnetb0_64_ft", FIGURES_DIR / "efficientnetb0_64_ft_confusion_matrix.png"
    ),
    "efficientnetb0_224_ft_b128_confusion": lambda: plot_confusion_matrix(
        "efficientnetb0_224_ft_b128",
        FIGURES_DIR / "efficientnetb0_224_ft_b128_confusion_matrix.png",
    ),
}


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    group = parser.add_mutually_exclusive_group(required=True)
    group.add_argument("--all", action="store_true", help="build every report figure")
    group.add_argument("--figure", choices=sorted(BUILDERS), help="build one figure")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    names = sorted(BUILDERS) if args.all else [args.figure]
    for name in names:
        path = BUILDERS[name]()
        print(f"[figures] {name} -> {path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Source: `scripts/build_report.py`


In [ ]:
%%writefile scripts/build_report.py
#!/usr/bin/env python
"""Build the report PDF from report/main.tex in one command.

    python scripts/build_report.py            # -> report/main.pdf
    python scripts/build_report.py --clean    # remove LaTeX build artifacts

Runs latexmk, which reruns pdflatex until every cross-reference resolves.
Works the same on Windows (MiKTeX), Linux and macOS (TeX Live); CI compiles
the same file with xu-cheng/latex-action, so a report that builds here builds
there. The tables and figures it includes are not regenerated here - run
scripts/build_tables.py and scripts/build_figures.py first if results changed.
"""

from __future__ import annotations

import argparse
import shutil
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))

from pr_edge.paths import REPORT_DIR  # noqa: E402

MAIN_TEX = REPORT_DIR / "main.tex"


def latexmk_command(clean: bool = False) -> list[str]:
    """The latexmk invocation, run from inside report/.

    Running from report/ matters: main.tex and every section file use paths
    relative to it (tables/..., figures/...). -halt-on-error makes a broken
    \\input fail the build loudly instead of producing a half-built PDF.
    """
    if clean:
        return ["latexmk", "-c", MAIN_TEX.name]
    return ["latexmk", "-pdf", "-interaction=nonstopmode", "-halt-on-error", MAIN_TEX.name]


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--clean", action="store_true", help="remove build artifacts, keep main.pdf")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    if shutil.which("latexmk") is None:
        print("[report] latexmk not found. Install a LaTeX distribution "
              "(MiKTeX on Windows, TeX Live on Linux/macOS) and retry.", file=sys.stderr)
        return 1

    result = subprocess.run(latexmk_command(args.clean), cwd=REPORT_DIR)
    if result.returncode != 0:
        print(f"[report] latexmk failed - see {REPORT_DIR / 'main.log'}", file=sys.stderr)
        return result.returncode
    if not args.clean:
        print(f"[report] built {REPORT_DIR / 'main.pdf'}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Run / inspect results


In [ ]:
import json
from IPython.display import Image, display
from pr_edge.schemas import validate_run_dir

for name in ("model_a", "model_a_100", "model_b", "model_b_100",
             "model_b_sgd", "model_b_sgdm", "mobilenetv2_64_ft", "efficientnetb0_64_ft"):
    folder = PROJECT / "results" / name
    errors = validate_run_dir(folder)
    assert not errors, (name, errors)
    metrics = json.loads((folder / "metrics.json").read_text())
    profile = json.loads((folder / "profile.json").read_text())
    print(name, "accuracy:", metrics["accuracy"],
          "parameters:", profile["total_params"], "MACs:", profile["macs"])

# Rebuild from the supplied recorded JSON; no training or dataset download.
run_script("scripts/build_tables.py", "--all")
run_script("scripts/build_figures.py", "--all")
display(Image(filename=str(PROJECT / "report/figures/custom_loss_curves.png")))
display(Image(filename=str(PROJECT / "report/figures/efficientnetb0_64_ft_confusion_matrix.png")))

# Optional PDF rebuild requires a local LaTeX installation and latexmk.
# The final PDF is supplied separately from this code ZIP.
BUILD_PDF = False
if BUILD_PDF:
    run_script("scripts/build_report.py")
